In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

RAW = Path('data/raw')
PROCESSED = Path('data/cleaned_data')
PROCESSED.mkdir(parents=True, exist_ok=True)

## Phase 1: Loading the raw data

### CBS tables
Both CBS tables contain all regions (country, municipalities, districts, neighbourhoods). The codes in `Codering_3` and `SoortRegio_2` have trailing spaces, which would make merges fail silently, so we strip them.

In [2]:
# Load the raw data

kwb = pd.read_csv(RAW / 'kwb_2023.csv', low_memory=False)
nabijheid = pd.read_csv(RAW / 'nabijheid_2023.csv', low_memory=False)

for df in (kwb, nabijheid):
    df['Codering_3'] = df['Codering_3'].str.strip()
    df['SoortRegio_2'] = df['SoortRegio_2'].str.strip()

print('kwb:      ', kwb.shape)
print('nabijheid:', nabijheid.shape)
kwb['SoortRegio_2'].value_counts()


kwb:       (18116, 128)
nabijheid: (18116, 116)


SoortRegio_2
Buurt       14421
Wijk         3352
Gemeente      342
Land            1
Name: count, dtype: int64

In [3]:
# Check that the CBS tables match on region code
niet_gematcht = ~kwb['Codering_3'].isin(nabijheid['Codering_3'])
print('Regions in kwb without match in nabijheid:', niet_gematcht.sum())

Regions in kwb without match in nabijheid: 0


In [4]:
# Merge the CBS tables on region code        

stations = pd.read_csv(RAW / 'stations-2023-09-nl.csv')

storingen = pd.concat(
    [pd.read_csv(RAW / f'disruptions-{jaar}.csv') for jaar in [2021, 2022, 2023]],
    ignore_index=True,
)
storingen['start_time'] = pd.to_datetime(storingen['start_time'])
storingen['end_time'] = pd.to_datetime(storingen['end_time'])

print('stations: ', stations.shape)
print('storingen:', storingen.shape)

stations:  (397, 11)
storingen: (15541, 14)


In [5]:
# Load the CBS map of neighborhoods


buurten_kaart = gpd.read_file(
    RAW / 'wijkenbuurten_2023_v3.gpkg',
    layer='buurten',
    columns=['buurtcode', 'buurtnaam', 'gemeentenaam'],
)
print(buurten_kaart.crs, buurten_kaart.shape)

c:\Users\alecm\anaconda3\envs\TIL6022-26\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


EPSG:28992 (14515, 4)


## Phase 2: Neighbourhood sample filtering

### Select and rename variables
We keep only the relevant columns and give them short names.

In [6]:
#Kolom namen van de CBS tabellen aanpassen en samenvoegen

kwb_kolommen = {
    'Codering_3': 'buurtcode',
    'WijkenEnBuurten': 'buurtnaam',
    'Gemeentenaam_1': 'gemeente',
    'SoortRegio_2': 'soort_regio',
    'PersonenautoSPerHuishouden_112': 'autos_per_hh',
    'GemiddeldInkomenPerInwoner_81': 'inkomen',
    'MateVanStedelijkheid_125': 'stedelijkheid',
    'GemiddeldeHuishoudensgrootte_33': 'hh_grootte',
    'Bevolkingsdichtheid_34': 'dichtheid',
    'HuishoudensTotaal_29': 'n_huishoudens',
    'BedrijfsvestigingenTotaal_100': 'n_bedrijven',
}
nabijheid_kolommen = {
    'Codering_3': 'buurtcode',
    'AfstandTotTreinstationsTotaal_90': 'afstand_station',
    'AfstandTotBelangrijkOverstapstation_91': 'afstand_overstapstation',
}

df = (kwb[list(kwb_kolommen)].rename(columns=kwb_kolommen)
      .merge(nabijheid[list(nabijheid_kolommen)].rename(columns=nabijheid_kolommen),
             on='buurtcode', how='left'))
df.head()

,buurtcode,buurtnaam,gemeente,soort_regio,autos_per_hh,inkomen,stedelijkheid,hh_grootte,dichtheid,n_huishoudens,n_bedrijven,afstand_station,afstand_overstapstation
0,NL00,Nederland,Nederland,Land,1.1,32.8,2.0,2.1,529.0,8270244,2358605,5.3,10.8
1,GM1680,Aa en Hunze,Aa en Hunze,Gemeente,1.4,32.8,5.0,2.2,93.0,11327,3180,12.4,14.4
2,WK168000,Wijk 00 Annen,Aa en Hunze,Wijk,1.3,34.1,5.0,2.3,246.0,1585,405,13.3,14.3
3,BU16800000,Annen,Aa en Hunze,Buurt,1.3,33.4,5.0,2.3,1369.0,1520,375,13.2,14.2
4,BU16800009,Verspreide huizen Annen,Aa en Hunze,Buurt,1.5,49.5,5.0,2.3,12.0,65,30,13.7,15.3


In [7]:
# Filter the data to only include neighborhoods (buurten) and remove rows with missing values for autos_per_hh

df = df[df['soort_regio'] == 'Buurt']
df = df[df['autos_per_hh'].notna()]

modelvariabelen = ['afstand_station', 'inkomen', 'stedelijkheid', 'hh_grootte', 'dichtheid']
df = df.dropna(subset=modelvariabelen)

In [8]:
# Calculate the number of businesses per household (commercial zones where corporate leasing fleets artificially inflate car counts)

df['bedrijven_per_hh'] = df['n_bedrijven'] / df['n_huishoudens']

(df.sort_values('bedrijven_per_hh', ascending=False)
   [['buurtnaam', 'gemeente', 'n_huishoudens', 'bedrijven_per_hh', 'autos_per_hh']]
   .head(15))

,buurtnaam,gemeente,n_huishoudens,bedrijven_per_hh,autos_per_hh
6428,Hoendiep,Groningen,60,4.166667,2.2
7518,Schooten,Helmond,105,3.285714,2.2
1519,Zuidas-Noord,Amsterdam,355,3.000000,0.6
2806,'t Zand,Best,110,2.909091,2.0
1743,Amstel III deel A/B-Noord,Amsterdam,395,2.746835,1.8
7730,De Rietvelden oost,'s-Hertogenbosch,85,2.470588,1.6
6388,Eemskanaal,Groningen,110,2.409091,1.9
6002,Tijlvoort-Noort,Goirle,50,2.400000,1.8
557,Gooisekant Oost,Almere,115,2.347826,1.8
17029,Bedrijventerrein Vlotlaan,Westland,95,2.315789,2.4


In [9]:
# Filter out neighborhoods with an unusually high number of businesses per household (we don't want to include commercial zones in our analysis)

DREMPEL = 1.0
df = df[df['bedrijven_per_hh'] <= DREMPEL]

print('Aantal buurten over:', len(df))

Aantal buurten over: 11221


In [11]:
%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [12]:
# making clean dataframes and saving them to disk

df = df.drop(columns='soort_regio', errors='ignore').reset_index(drop=True)

# pickles: used by the next notebooks
df.to_pickle(PROCESSED / 'buurten_schoon.pkl')
stations.to_pickle(PROCESSED / 'stations.pkl')
storingen.to_pickle(PROCESSED / 'storingen.pkl')
buurten_kaart.to_file(PROCESSED / 'buurten_kaart.gpkg', driver='GPKG')

# Excel: only for viewing
df.to_excel(PROCESSED / 'buurten_schoon.xlsx', index=False)
stations.to_excel(PROCESSED / 'stations.xlsx', index=False)

for f in sorted(PROCESSED.iterdir()):
    print(f'{f.name:25s} {f.stat().st_size / 1e6:6.1f} MB')

buurten_kaart.gpkg          85.2 MB
buurten_schoon.pkl           1.9 MB
buurten_schoon.xlsx          0.9 MB
stations.pkl                 0.1 MB
stations.xlsx                0.0 MB
storingen.pkl                5.3 MB
